## SIT319 Assignment 2 
### Regularisation, Capacity, and Generalisation


##### Author: Tina Kumari


### 1. Import Libraries

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

ModuleNotFoundError: No module named 'torch'

### 2. Experimental Setup  
#### 2.1 Dataset Preparation

The Breast Cancer Wisconsin (Diagnostic) dataset is loaded and its integrity is verified before use.

The dataset consists of:
- a feature matrix (X) containing 30 numerical features per sample  
- a target vector (y) representing a binary classification problem (0 and 1)  

Sanity checks are performed to confirm that:
- X and y are NumPy arrays  
- X is two-dimensional and y is one-dimensional  
- the number of samples in X and y match  
- the feature dimension is 30  
- the target values are strictly binary  

These checks ensure that the dataset is valid and suitable for subsequent model training and evaluation.

In [ ]:
data = load_breast_cancer()
X = data.data.astype(np.float32)
y = data.target.astype(np.int64)

# Dataset integrity checks
assert isinstance(X, np.ndarray), "X must be a NumPy array"
assert isinstance(y, np.ndarray), "y must be a NumPy array"
assert X.ndim == 2, "X must be 2-dimensional"
assert y.ndim == 1, "y must be 1-dimensional"
assert X.shape[0] == y.shape[0], "Mismatch between X and y samples"
assert X.shape[1] == 30, "Feature dimension must be 30"
assert set(np.unique(y)) == {0, 1}, "Target must be binary (0 and 1)"

print("+++ Dataset checks passed.")
print("X shape:", X.shape)
print("y shape:", y.shape)
print("Class counts:", np.bincount(y))

### 2.2 Train / Validation / Test Split

The dataset is divided into training, validation, and test sets using a stratified split with a random seed of 42. The proportions are approximately 70% for training, 15% for validation, and 15% for testing. Stratification ensures that the class distribution is preserved across all splits, which is important for reliable evaluation in a binary classification task.

The role of each subset is as follows:
- The training set is used to learn model parameters  
- The validation set is used to monitor performance and guide model comparison  
- The test set is reserved for final evaluation of generalisation  

After splitting, checks are performed to ensure that:
- the total number of samples is preserved  
- feature and label counts match in each subset  
- all subsets contain both target classes  

This controlled split provides a reliable basis for analysing overfitting, regularisation, and generalisation.

In [ ]:
# Split into train (70%) and temp (30%)
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=42,
    stratify=y
)

# Split temp into validation (15%) and test (15%)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.5,
    random_state=42,
    stratify=y_temp
)

# Check split integrity
total = X.shape[0]
n_train = X_train.shape[0]
n_val = X_val.shape[0]
n_test = X_test.shape[0]

assert n_train + n_val + n_test == total, "Split sizes do not sum to total"
assert X_train.shape[0] == y_train.shape[0]
assert X_val.shape[0] == y_val.shape[0]
assert X_test.shape[0] == y_test.shape[0]
assert set(np.unique(y_train)) == {0, 1}, "Train set must contain both classes"
assert set(np.unique(y_val)) == {0, 1}, "Validation set must contain both classes"
assert set(np.unique(y_test)) == {0, 1}, "Test set must contain both classes"

print("+++ Data split checks passed.")

#### Feature Scaling

The input features are standardised using `StandardScaler`.

The scaler is fitted only on the training set, and then the same fitted transformation is applied to the validation and test sets. This prevents data leakage, because no information from the validation or test data is used when computing the scaling parameters.

Standardisation gives each feature approximately zero mean and unit variance, which helps neural networks train more stably and prevents features with larger numeric ranges from dominating the learning process.

In [ ]:
# Feature scaling
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train).astype(np.float32)
X_val = scaler.transform(X_val).astype(np.float32)
X_test = scaler.transform(X_test).astype(np.float32)

print("Scaled shapes:")
print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape) 

### 2.3 Baseline Architecture

The dataset is converted into PyTorch tensors and organised using `TensorDataset` and `DataLoader` to enable mini-batch training. A batch size of 64 is used, with shuffling applied only to the training set.

Two fully connected MLP architectures are defined. The small model consists of a single hidden layer with 64 units, while the large model contains three hidden layers with 256, 128, and 64 units. ReLU activation functions are applied after each hidden layer, and the output layer produces logits for binary classification.

Both models are trained using the Adam optimiser with a learning rate of 1e-3 and cross-entropy loss for 60 epochs. A PyTorch random seed of 42 is set before model initialisation to ensure reproducibility. The number of trainable parameters is also computed to highlight the difference in model capacity.

In [ ]:
# Convert NumPy arrays to PyTorch tensors
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)

X_val_tensor = torch.tensor(X_val, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val, dtype=torch.long)

X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

# Create datasets
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
val_dataset = TensorDataset(X_val_tensor, y_val_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

# Create DataLoaders for mini-batch training
batch_size = 64

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))

In [ ]:
class SmallMLP(nn.Module):
    def __init__(self, input_dim, num_classes=2):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        return self.network(x)

class LargeMLP(nn.Module):
    def __init__(self, input_dim, num_classes=2):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        return self.network(x)

# Set seed and initialise models
torch.manual_seed(42)
small_model = SmallMLP(input_dim=X_train.shape[1])

torch.manual_seed(42)
large_model = LargeMLP(input_dim=X_train.shape[1])

# Loss function
criterion = nn.CrossEntropyLoss()

# Training configuration
learning_rate = 1e-3
num_epochs = 60

# Optimisers
optimizer_small = torch.optim.Adam(small_model.parameters(), lr=learning_rate)
optimizer_large = torch.optim.Adam(large_model.parameters(), lr=learning_rate)

print("Models and training configuration set up successfully.")
print("Small model parameters:", sum(p.numel() for p in small_model.parameters()))
print("Large model parameters:", sum(p.numel() for p in large_model.parameters()))

### 2.4 Required Training Output Scheme


For each model, training is performed for 60 epochs. During training, the following metrics are recorded at each epoch in a history dictionary:
- `train_loss`: training loss  
- `val_loss`: validation loss  
- `train_acc`: training accuracy  
- `val_acc`: validation accuracy  

After training, the final training and validation accuracies are taken from the last epoch. The test set is then evaluated once to obtain the final test accuracy.

The generalisation gap is computed as:

generalisation_gap = final_train_acc - final_val_acc

This measures the difference between training and validation performance. A larger gap indicates stronger overfitting, while a smaller gap suggests better generalisation.

The test set is used only after training is complete and is not involved in model selection or tuning.

In [ ]:
# Calculates accuracy of a model on a given dataset
def compute_accuracy(model, dataloader):
    model.eval()  

    correct = 0  
    total = 0    
    
    # Disable gradient computation (not needed during evaluation hence faster and less memory)
    with torch.no_grad(): 
        for X_batch, y_batch in dataloader:  

            outputs = model(X_batch)  # Forward pass: get model predictions (logits)
            
            # Convert logits to predicted class (0 or 1) by selecting highest score
            predictions = torch.argmax(outputs, dim=1) 
            
            # Count how many predictions match the true labels
            correct += (predictions == y_batch).sum().item()  
            
            # Add number of samples in this batch
            total += y_batch.size(0)  

    return correct / total  

# Calculates average loss of the model on a dataset (used for validation)
def evaluate_loss(model, dataloader, criterion):
    model.eval()  

    total_loss = 0.0      
    total_samples = 0   

    # Disable gradients since we are not training
    with torch.no_grad():
        for X_batch, y_batch in dataloader:

            outputs = model(X_batch)  
            loss = criterion(outputs, y_batch)

            batch_size = y_batch.size(0)  

            # Multiply loss by batch size to accumulate correctly
            total_loss += loss.item() * batch_size  

            total_samples += batch_size

    # Return average loss across entire dataset
    return total_loss / total_samples

In [ ]:
# Trains the model and records training/validation metrics
def train_model(model, train_loader, val_loader, criterion, optimizer, num_epochs=60):

    # Dictionary to store results for each epoch (required format)
    history = {
        "train_loss": [],
        "val_loss": [],
        "train_acc": [],
        "val_acc": [],
    }

    # Loop over epochs (full passes through the training data)
    for epoch in range(num_epochs):

        model.train()  # Set model to training mode (enables learning)
        
        running_loss = 0.0  
        total_samples = 0    

        # Loop through training data in batches
        for X_batch, y_batch in train_loader:
            # Reset gradients from previous step 
            optimizer.zero_grad()  
            # Forward pass: compute predictions
            outputs = model(X_batch)  
            # Compute loss (difference between predictions and true labels)
            loss = criterion(outputs, y_batch)  
            # Backpropagation: compute gradients (how to fix weights)
            loss.backward()  
            # Update model weights using gradients
            optimizer.step()  
            # Accumulate total loss (scaled by batch size)
            batch_size = y_batch.size(0)

            running_loss += loss.item() * batch_size  
            total_samples += batch_size  # Count samples

        # Compute average training loss for this epoch
        train_loss = running_loss / total_samples  

        # Compute validation loss using helper function
        val_loss = evaluate_loss(model, val_loader, criterion)

        # Compute accuracy on training and validation sets
        train_acc = compute_accuracy(model, train_loader)
        val_acc = compute_accuracy(model, val_loader)

        # Store results in history dictionary
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)

    # Final metrics = last epoch values
    final_train_acc = history["train_acc"][-1]
    final_val_acc = history["val_acc"][-1]

    # Return all recorded metrics + final accuracies
    return history, final_train_acc, final_val_acc

In [ ]:
# Set seed for reproducibility (same weights every run)
torch.manual_seed(42)

# Initialise small model (input_dim = number of features = 30)
small_model = SmallMLP(input_dim=X_train.shape[1])

# Define optimizer (Adam) with learning rate 1e-3
optimizer_small = torch.optim.Adam(small_model.parameters(), lr=1e-3)

# Train the small model for 60 epochs
history_small, final_train_acc_small, final_val_acc_small = train_model(
    small_model,
    train_loader,
    val_loader,
    criterion,
    optimizer_small,
    num_epochs=60
)

# Evaluate final performance on test set (only once after training)
final_test_acc_small = compute_accuracy(small_model, test_loader)

# Compute generalisation gap (difference between train and validation accuracy)
generalisation_gap_small = final_train_acc_small - final_val_acc_small

# Print results for small model
print("Small model results:")
print("Final train accuracy:", final_train_acc_small)
print("Final validation accuracy:", final_val_acc_small)
print("Final test accuracy:", final_test_acc_small)
print("Generalisation gap:", generalisation_gap_small)

In [ ]:
# Set seed again for fair comparison 
torch.manual_seed(42)

# Initialise large model (higher capacity)
large_model = LargeMLP(input_dim=X_train.shape[1])

# Define optimizer for large model
optimizer_large = torch.optim.Adam(large_model.parameters(), lr=1e-3)

# Train the large model for 60 epochs
history_large, final_train_acc_large, final_val_acc_large = train_model(
    large_model,
    train_loader,
    val_loader,
    criterion,
    optimizer_large,
    num_epochs=60
)

# Evaluate final performance on test set
final_test_acc_large = compute_accuracy(large_model, test_loader)

# Compute generalisation gap
generalisation_gap_large = final_train_acc_large - final_val_acc_large

# Print results for large model
print("Large model results:")
print("Final train accuracy:", final_train_acc_large)
print("Final validation accuracy:", final_val_acc_large)
print("Final test accuracy:", final_test_acc_large)
print("Generalisation gap:", generalisation_gap_large)

#### Training Results and Generalisation

Both the small and large models were trained for 60 epochs, and their final performance metrics were recorded.

The small model achieved a training accuracy of approximately 0.99 and a validation accuracy of 0.976, resulting in a small generalisation gap of 0.013. This suggests good generalisation, as performance on unseen data closely matches training performance.

The large model achieved perfect training accuracy (1.00), indicating that it has sufficient capacity to fully fit the training data. However, its validation accuracy remained similar (0.976), resulting in a larger generalisation gap of 0.023, which indicates overfitting.

Both models achieved similar test accuracy (approximately 0.965), showing that increasing model capacity did not improve performance on unseen data.

Overall, these results demonstrate that while larger models have higher capacity and optimise training performance more effectively, they are more prone to overfitting and do not necessarily provide better generalisation.

#### Training Result Checks

Validation checks are performed to ensure the results follow the required output schema.

These checks confirm that all required metrics are present, each contains 60 entries (one per epoch), and that the final accuracies and generalisation gap are within valid ranges.

This ensures the training process has been implemented correctly.

In [ ]:
# Required keys that must exist in the history dictionary
required_keys = ["train_loss", "val_loss", "train_acc", "val_acc"]
n_epochs = 60

# Small model checks
for k in required_keys:
    assert k in history_small, f"Missing key: {k}"
    assert len(history_small[k]) == n_epochs, f"{k} must contain {n_epochs} entries"

assert 0 <= final_train_acc_small <= 1
assert 0 <= final_val_acc_small <= 1
assert 0 <= final_test_acc_small <= 1
assert -0.5 <= generalisation_gap_small <= 0.5

print("+++ Small model training result checks passed.")

# Large model checks
for k in required_keys:
    assert k in history_large, f"Missing key: {k}"
    assert len(history_large[k]) == n_epochs, f"{k} must contain {n_epochs} entries"

assert 0 <= final_train_acc_large <= 1
assert 0 <= final_val_acc_large <= 1
assert 0 <= final_test_acc_large <= 1
assert -0.5 <= generalisation_gap_large <= 0.5

print("+++ Large model training result checks passed.")

## 3. Capacity and Generalisation Study

This section investigates how model capacity influences training performance, validation performance, and the generalisation gap.

Model capacity refers to the expressive power of the network and is controlled by the number of hidden layers, hidden units, and total number of trainable parameters. The large model therefore represents a higher-capacity network compared to the small model.

To isolate the effect of capacity, both models are trained using the same fixed configuration defined earlier, with no L2 regularisation applied.

For each model, the following metrics are recorded:
- final training accuracy  
- final validation accuracy  
- final test accuracy  
- generalisation gap  

Training and validation loss curves are also plotted across epochs to compare learning behaviour.

#### Task 3.1 — Models to Train

In [ ]:
epochs = range(1, 61)

plt.figure(figsize=(8, 5))
plt.plot(epochs, history_small["train_loss"], label="Training Loss")
plt.plot(epochs, history_small["val_loss"], label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Small Model: Training vs Validation Loss")
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(epochs, history_large["train_loss"], label="Training Loss")
plt.plot(epochs, history_large["val_loss"], label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Large Model: Training vs Validation Loss")
plt.legend()
plt.show()

### 3.2 Structured Analysis
#### Task 3.1 — Overfitting Behaviour

The large model achieves higher training accuracy, reaching 1.00, compared to the small model (0.99).It also has a larger generalisation gap (~0.024 vs ~0.013), indicating stronger overfitting.

From the loss curves, the large model’s training loss continues to decrease towards zero, while its validation loss begins to increase after approximately epoch 8–10. This divergence indicates the onset of overfitting.

In contrast, the small model shows closely aligned training and validation loss curves throughout training, suggesting better generalisation due to its lower capacity.


#### Task 3.2 — Capacity Interpretation

Increasing model capacity improves training performance because a higher-capacity model has greater flexibility to represent complex functions. This is achieved through a larger number of parameters and deeper architecture, allowing the model to closely fit the training data.

However, this increased flexibility also raises the risk of overfitting, as the model may begin to memorise noise or specific patterns in the training data rather than learning general features.

In this study, the large model, with significantly more parameters than the small model, achieved perfect training accuracy but did not improve validation or test accuracy. This resulted in a larger generalisation gap, demonstrating that increased capacity can harm generalisation despite improved optimisation.

### Sanity Checks 

Additional sanity checks are performed to confirm that training loss decreased during optimisation and that no L2 regularisation was used in this section.

In [ ]:
# Small model capacity study checks
history = history_small
optimizer = optimizer_small

# ---- Training progress sanity ----
# Check that training loss decreased from the first epoch to the last epoch
assert history["train_loss"][0] > history["train_loss"][-1], \
"Training loss did not decrease"

# ---- Ensure no L2 regularisation used ----
# weight_decay must be 0.0 in this section
assert optimizer.param_groups[0]["weight_decay"] == 0.0, \
"L2 regularisation is not allowed in this section"

print("+++ Small model capacity study checks passed.")

In [ ]:
# Large model capacity study checks
history = history_large
optimizer = optimizer_large

# ---- Training progress sanity ----
# Check that training loss decreased from the first epoch to the last epoch
assert history["train_loss"][0] > history["train_loss"][-1], \
"Training loss did not decrease"

# ---- Ensure no L2 regularisation used ----
# weight_decay must be 0.0 in this section
assert optimizer.param_groups[0]["weight_decay"] == 0.0, \
"L2 regularisation is not allowed in this section"

print("+++ Large model capacity study checks passed.")



### 4. L2 Regularisation Study

In Section 3, the effect of model capacity on generalisation was studied without explicit regularisation. In this section, the effect of L2 regularisation is isolated using the large-capacity model.

L2 regularisation reduces overfitting by penalising large parameter values. It adds a term to the loss function proportional to the squared magnitude of the model weights, encouraging smaller parameters and smoother decision boundaries.

In PyTorch, L2 regularisation is implemented through the `weight_decay` parameter of the Adam optimiser, where the regularisation strength is controlled by λ (`weight_decay = λ`). This setting is applied to all trainable parameters, including biases.

#### 4.2 Models to Train (Large Model Only)

The large-capacity model from Section 3 is used, consisting of three hidden layers with 256, 128, and 64 units and ReLU activations. The training configuration remains unchanged, using cross-entropy loss, the Adam optimiser, a learning rate of 1e-3, batch size of 64, and 60 epochs.

To evaluate the effect of L2 regularisation, the same model is trained using three values of λ:
- λ = 0.0 (no regularisation)  
- λ = 1e-4 (weak regularisation)  
- λ = 1e-2 (strong regularisation)  

L2 regularisation is applied through the `weight_decay` parameter in the optimiser for all trainable parameters.

In [ ]:
# Define L2 regularisation strengths to test
lambdas = [0.0, 1e-4, 1e-2]

# Dictionary to store results for each lambda value
results_l2 = {}

# Loop through each lambda value
for lam in lambdas:
    print(f"\nTraining with lambda = {lam}")

    # Set random seed for reproducibility 
    torch.manual_seed(42)

    # Create a fresh large model for this run
    model = LargeMLP(input_dim=X_train.shape[1])

    # Define Adam optimizer with L2 regularisation via weight_decay
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=1e-3,
        weight_decay=lam  # L2 regularisation strength (lambda)
    )

    # Train the model for 60 epochs
    history, final_train_acc, final_val_acc = train_model(
        model,
        train_loader,
        val_loader,
        criterion,
        optimizer,
        num_epochs=60
    )

    # Evaluate final test accuracy after training is complete
    final_test_acc = compute_accuracy(model, test_loader)

    # Compute generalisation gap 
    generalisation_gap = final_train_acc - final_val_acc

    # Store all results for this lambda value
    results_l2[lam] = {
        "history": history,  # Loss and accuracy over epochs
        "final_train_acc": final_train_acc,
        "final_val_acc": final_val_acc,
        "final_test_acc": final_test_acc,
        "generalisation_gap": generalisation_gap,
    }

    # Print results for this lambda run
    print("Final train accuracy:", final_train_acc)
    print("Final val accuracy:", final_val_acc)
    print("Final test accuracy:", final_test_acc)
    print("Generalisation gap:", generalisation_gap)

#### L2 Regularisation Results

The effect of L2 regularisation was evaluated using three values of λ.

For λ = 0.0 and λ = 1e-4, the results are nearly identical, indicating that weak regularisation has negligible impact on model behaviour. The model achieves perfect training accuracy and exhibits a generalisation gap of approximately 0.0235.

For λ = 1e-2, training accuracy decreases slightly, while validation and test accuracy remain unchanged. The generalisation gap reduces to approximately 0.0185, indicating reduced overfitting.

These results show that stronger L2 regularisation constrains the model by limiting parameter magnitude, reducing overfitting and improving generalisation behaviour, even when predictive performance remains unchanged.

### 4.3 Required Outputs Per Run

For each value of λ, the same history structure is recorded as in previous sections.

In addition, the following scalar metrics are computed:
- final training accuracy  
- final validation accuracy  
- final test accuracy  
- generalisation gap  

The generalisation gap measures the difference between training and validation performance.

The weight norm is also calculated as the global L2 norm of all trainable parameters (including weights and biases). This helps assess how L2 regularisation affects the overall magnitude of the model parameters.

In [ ]:
# Computes the global L2 norm of all trainable parameters in the model
def compute_weight_norm(model):
    total_squared_sum = 0.0  # Accumulator for sum of squared parameters

    # Loop through all trainable parameters (weights and biases)
    for param in model.parameters():
        # Square each parameter value, sum them, and add to total
        total_squared_sum += torch.sum(param ** 2).item()

    # Take square root of the total sum to get L2 norm
    weight_norm = total_squared_sum ** 0.5

    return weight_norm  # Return final scalar value

In [ ]:
# Define L2 regularisation strengths to test
lambdas = [0.0, 1e-4, 1e-2]

# Dictionary to store results for each lambda value
results_l2 = {}

# Loop through each lambda value
for lam in lambdas:
    print(f"\nTraining with lambda = {lam}")

    # Set random seed for reproducibility
    torch.manual_seed(42)

    # Create a fresh large model
    model = LargeMLP(input_dim=X_train.shape[1])

    # Define Adam optimizer with L2 regularisation
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=1e-3,
        weight_decay=lam
    )

    # Train the model
    history, final_train_acc, final_val_acc = train_model(
        model,
        train_loader,
        val_loader,
        criterion,
        optimizer,
        num_epochs=60
    )

    # Evaluate final test accuracy
    final_test_acc = compute_accuracy(model, test_loader)

    # Compute generalisation gap
    generalisation_gap = final_train_acc - final_val_acc

    # Compute global weight norm over all trainable parameters
    weight_norm = compute_weight_norm(model)

    # Store all results for this lambda
    results_l2[lam] = {
        "history": history,
        "final_train_acc": final_train_acc,
        "final_val_acc": final_val_acc,
        "final_test_acc": final_test_acc,
        "generalisation_gap": generalisation_gap,
        "weight_norm": weight_norm,
    }

    # Print results
    print("Final train accuracy:", final_train_acc)
    print("Final val accuracy:", final_val_acc)
    print("Final test accuracy:", final_test_acc)
    print("Generalisation gap:", generalisation_gap)
    print("Weight norm:", weight_norm)

#### Effect of L2 Regularisation on Weight Norm and Generalisation

The results show a clear relationship between L2 regularisation strength and the magnitude of model parameters. As λ increases, the weight norm decreases significantly, from approximately 14.08 (λ = 0.0) to 3.77 (λ = 1e-2). This confirms that L2 regularisation effectively constrains the size of the model weights.

At the same time, increasing λ slightly reduces training accuracy and decreases the generalisation gap, indicating reduced overfitting. However, validation and test accuracy remain unchanged across all values of λ.

Overall, these results demonstrate that L2 regularisation helps control model complexity by limiting parameter magnitude, leading to improved generalisation behaviour even when predictive performance remains similar.

### 4.4 Aggregated Summary Table

An aggregated summary table is presented to compare the effect of different L2 regularisation strengths (λ) on model performance and weight magnitude.

In [ ]:
# Create a list to store rows
summary_data = []

# Loop through results dictionary
for lam, res in results_l2.items():
    summary_data.append({
        "lambda": lam,
        "final_train_acc": res["final_train_acc"],
        "final_val_acc": res["final_val_acc"],
        "final_test_acc": res["final_test_acc"],
        "generalisation_gap": res["generalisation_gap"],
        "weight_norm": res["weight_norm"],
    })

# Convert to DataFrame
summary_table = pd.DataFrame(summary_data)

# Sort by lambda 
summary_table = summary_table.sort_values(by="lambda")

# Display table
summary_table.round(4)

#### Summary Results

The results show that as λ increases, the weight norm decreases significantly, indicating that L2 regularisation effectively constrains the magnitude of model parameters. This reduction is most pronounced at λ = 1e-2, demonstrating strong parameter shrinkage.

At the same time, training accuracy slightly decreases, while the generalisation gap is reduced, indicating improved generalisation and reduced overfitting. Validation and test accuracy remain largely unchanged across all values of λ.

These findings suggest that L2 regularisation primarily controls model complexity rather than improving raw predictive performance. Overall, it acts as an effective form of norm control, stabilising training and reducing overfitting without significantly degrading performance.

### 4.5 Loss Curves with L2 Regularisation

Training and validation loss curves are plotted for each value of λ on the same figure. 

Different colours are used to distinguish between λ values, while line styles differentiate between training and validation loss. This allows comparison of how L2 regularisation affects learning behaviour over epochs.

In [ ]:
colors = {
    0.0: "blue",
    1e-4: "green",
    1e-2: "red"
}

epochs = range(1, 61)

plt.figure(figsize=(11, 6))

for lam, res in results_l2.items():
    history = res["history"]

    plt.plot(
        epochs,
        history["train_loss"],
        color=colors[lam],
        linestyle='-',
        linewidth=2,
        alpha=0.9,
        label=f"Train, λ={lam}"
    )

    plt.plot(
        epochs,
        history["val_loss"],
        color=colors[lam],
        linestyle='--',
        linewidth=2,
        alpha=0.9,
        label=f"Val, λ={lam}"
    )

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training and Validation Loss for Different L2 Regularisation Strengths")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

#### Loss Curve Observations

The training loss decreases steadily for all values of λ, indicating successful learning across all runs. 

For λ = 0.0 and λ = 1e-4, the training and validation curves overlap closely, showing similar learning behaviour and indicating that very weak regularisation has little impact.

For λ = 1e-2, the training loss decreases more gradually, while the validation loss remains more stable. This suggests that stronger L2 regularisation reduces overfitting by limiting the model’s ability to memorise the training data.

Overall, increasing λ leads to smoother training behaviour and improved generalisation.

### 4.6 Structured Analysis
#### Task 4.1 — Regularisation Effects

As λ increases, the weight norm decreases significantly, from approximately 14.08 (λ = 0.0) to 12.37 (λ = 1e-4) and 3.77 (λ = 1e-2). This demonstrates that L2 regularisation effectively constrains parameter magnitudes.

The generalisation gap also decreases with increasing λ, reducing from 0.0235 (λ = 0.0 and 1e-4) to 0.0185 (λ = 1e-2), indicating reduced overfitting. Notably, λ = 1e-4 produces results almost identical to λ = 0.0, suggesting that this level of regularisation is too weak to meaningfully influence the model.

The main trade-off is between optimisation and generalisation. Higher λ slightly reduces training accuracy (from 1.00 to 0.995), reflecting reduced ability to perfectly fit the training data while improving generalisation.

#### Task 4.2 — Mechanism

L2 regularisation improves generalisation by encouraging the model to learn smaller parameter values. By penalising large weights, it prevents the model from relying too heavily on specific features or noise in the training data.

Smaller parameter values lead to smoother and less complex decision boundaries, reducing the risk of overfitting. As a result, the model focuses on general patterns rather than memorising the training data.

However, this introduces a trade-off between fitting the training data and performing well on unseen data. While smaller weights improve generalisation, they can slightly reduce the model’s ability to fit the training data perfectly, as reflected in the observed decrease in training accuracy at higher λ values.

### 4.7 Sanity Checks

To satisfy the required output schema for the L2 regularisation study, the results from the three runs are stored in a dictionary named `runs`. Each run contains the corresponding λ value, training history, final performance metrics, generalisation gap, and weight norm.

Sanity checks are then applied to verify that all required fields are present and that the stored test accuracies and weight norms are valid.

In [ ]:
# Store results in the required schema
runs = {
    "0": {
        "lambda": 0.0,
        "history": results_l2[0.0]["history"],
        "final_train_acc": results_l2[0.0]["final_train_acc"],
        "final_val_acc": results_l2[0.0]["final_val_acc"],
        "final_test_acc": results_l2[0.0]["final_test_acc"],
        "generalisation_gap": results_l2[0.0]["generalisation_gap"],
        "weight_norm": results_l2[0.0]["weight_norm"],
    },
    "1e-4": {
        "lambda": 1e-4,
        "history": results_l2[1e-4]["history"],
        "final_train_acc": results_l2[1e-4]["final_train_acc"],
        "final_val_acc": results_l2[1e-4]["final_val_acc"],
        "final_test_acc": results_l2[1e-4]["final_test_acc"],
        "generalisation_gap": results_l2[1e-4]["generalisation_gap"],
        "weight_norm": results_l2[1e-4]["weight_norm"],
    },
    "1e-2": {
        "lambda": 1e-2,
        "history": results_l2[1e-2]["history"],
        "final_train_acc": results_l2[1e-2]["final_train_acc"],
        "final_val_acc": results_l2[1e-2]["final_val_acc"],
        "final_test_acc": results_l2[1e-2]["final_test_acc"],
        "generalisation_gap": results_l2[1e-2]["generalisation_gap"],
        "weight_norm": results_l2[1e-2]["weight_norm"],
    },
}

In [ ]:
# Required run keys
run_keys = ["0", "1e-4", "1e-2"]

# Required fields inside each run
required_fields = [
    "lambda",
    "history",
    "final_train_acc",
    "final_val_acc",
    "final_test_acc",
    "generalisation_gap",
    "weight_norm",
]

# Check that runs is a dictionary
assert isinstance(runs, dict), "runs must be a dict"

# Check that each required run key exists
for key in run_keys:
    assert key in runs, f"Missing run key: {key}"
    r = runs[key]

    # Check that each run contains all required fields
    for field in required_fields:
        assert field in r, f"Missing field '{field}' in runs['{key}']"

# Check that test accuracy and weight norm are valid for each run
for key in run_keys:
    r = runs[key]

    te = float(r["final_test_acc"])
    assert 0.0 <= te <= 1.0, f"Test acc out of range for run {key}"

    wn = float(r["weight_norm"])
    assert wn > 0.0 and np.isfinite(wn), f"Invalid weight_norm for run {key}"

print("+++ L2 regularisation study checks passed.")